# LTR + Prompt Length Extension — Full Pipeline (v3, rebuilt)


## 1. Mount Google Drive (FIRST — before anything else)

Everything in this notebook lives under `/content/drive/MyDrive/ltr_extension`
from this point forward. This directory survives session restarts,
disconnects, and runtime changes — `/content` on its own does not.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}
!pwd

## 2. Confirm GPU is attached

In [ ]:
!nvidia-smi
import torch
print()
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Clone your prompt-aware-ltr repo (into the Drive-mounted path)

Clones [github.com/waizinmon/prompt-aware-ltr](https://github.com/waizinmon/prompt-aware-ltr) -- your
extension code (`scripts/`, `data/`, `checkpoints/`), not the vLLM engine itself.
vLLM is installed separately via `pip install vllm` in the next step. If this is
a re-run and the folder already exists from a previous session, this cell skips
the clone safely instead of erroring.

In [ ]:
import os
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')

if os.path.exists(repo_path):
    print(f"Repo already exists at {repo_path}, skipping clone.")
else:
    !git clone https://github.com/waizinmon/prompt-aware-ltr.git {repo_path}

%cd {repo_path}
!pwd
!ls

## 4. Install everything in ONE pinned step (fixes prior version conflicts)


In [ ]:
!pip uninstall -y torch torchvision torchaudio vllm
!pip install -q --no-cache-dir "transformers==4.44.2" "tokenizers==0.19.1" "vllm==0.6.1.post2" datasets scipy matplotlib scikit-learn huggingface_hub

print()
print("Installed versions:")
import transformers, tokenizers
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
import torch
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
import vllm
print("vllm:", vllm.__version__)
print()
print("NOTE: original paper used vllm==0.4.1. This environment uses a")
print("newer version due to Python 3.12 compatibility AND a broken")
print("transitive dependency (pyairports) in the 0.5.x guided-decoding")
print("feature. State this version deviation explicitly in your report.")

## 5. RESTART RUNTIME NOW (required once, after Cell 4)

`torch`, `transformers`, and `vllm` may already be partially imported
in memory from the install process (Cell 2 already imported the old
torch to check CUDA). A restart clears that cleanly -- it's required,
not optional, or the version check in the next cell can give a false
pass/fail.

**Runtime → Restart session**

After restarting, **skip Cells 1-4** (Drive is still mounted, packages
are still installed -- a restart clears Python memory, not installed
packages or Drive files) and continue from Cell 6 onward.

In [ ]:
# Run this AFTER restarting, to confirm you're back in the right place
# and everything from Cells 1-4 survived the restart correctly.
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}
!pwd

import torch, transformers, tokenizers, vllm
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("vllm:", vllm.__version__)
print("\nIf the versions above match Cell 4's output, you're good to continue.")

## 6. HuggingFace login

Needed for: (a) downloading the gated LMSYS-Chat-1M dataset, and
(b) downloading the gated Llama-3.1-8B-Instruct model later.

Before running this cell:
1. Accept the dataset terms: https://huggingface.co/datasets/lmsys/lmsys-chat-1m
2. Accept the model terms: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
3. Get a token (read access is enough): https://huggingface.co/settings/tokens

**Paste your token directly into the login() call below just this once.**
After this cell runs, every later cell reuses this login automatically
via `HfFolder.get_token()` -- you will not need to paste it again.

In [ ]:
from huggingface_hub import login
login(token="PASTE_YOUR_TOKEN_HERE")  # <-- replace this string only, then run

## 7. Verify the project scripts are present

Since Section 3 now clones `prompt-aware-ltr` directly, the pipeline scripts
already live in `scripts/` -- no manual upload needed.

**Required for the core pipeline (Sections 8-14):**
```
scripts/0a_build_dataset.py
scripts/0b_train_predictor.py
scripts/1_scoring_patch.py
scripts/scoring_patch_helper.py
scripts/2_kendalls_tau_eval.py
scripts/3_vllm_benchmark.py
scripts/4_plot_results.py
scripts/5_extended_metrics.py
scripts/6_hol_blocking_eval.py

```

In [ ]:
import os
required = ["0a_build_dataset.py", "0b_train_predictor.py", "1_scoring_patch.py",
            "scoring_patch_helper.py", "2_kendalls_tau_eval.py", "3_vllm_benchmark.py", "4_plot_results.py", "5_extended_metrics.py", "6_hol_blocking_eval.py"]

scripts_dir = os.path.join(repo_path, "scripts")
for fname in required:
    path = os.path.join(scripts_dir, fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"[{status}] {fname}")


**Note:** Sections 8–12 are the compute-heavy part of this pipeline
(building the dataset, training both predictor checkpoints, and running
the two vLLM benchmark sweeps), and the benchmark sweeps specifically
benefit from an A100. **If you don't have an A100 GPU runtime, skip
Sections 8–12 and run Section 13 directly** -- that assumes
`checkpoints/`, `data/`, and `result/` already have files in them from
a previous run on this Drive.

## 8. Build the dataset

Downloads the real public LMSYS-Chat-1M dataset and splits it into
training data + two evaluation sets. Dataset 2 deliberately samples
from the extremes of the output-length distribution to create genuine
distribution shift, matching what the original paper's Dataset 2 was
designed to expose.

**Note:** this reconstructs an equivalent dataset from the same public
source the original paper cites -- it is not their exact unpublished
23,800-sample file. Report results as "evaluated on a reconstructed
LMSYS-Chat-1M split," not as an exact reproduction.

This step is skipped automatically if the files already exist from a
previous run in this session.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

import os
from huggingface_hub import HfFolder

data_dir = os.path.join(repo_path, 'data')
expected_files = ['train_predictor.jsonl', 'dataset1_eval.jsonl', 'dataset2_eval.jsonl']
already_built = os.path.exists(data_dir) and all(
    os.path.exists(os.path.join(data_dir, f)) for f in expected_files
)

if already_built:
    print(f"Dataset files already exist in {data_dir}, skipping build.")
    !ls -la {data_dir}
else:
    token = HfFolder.get_token()
    !python scripts/0a_build_dataset.py \
        --hf_token {token} \
        --train_size 23800 \
        --eval_size 1000 \
        --output_dir ./data

## 9. Train the predictor — TWO checkpoints

Trains both the original paper's baseline (ListMLE loss) and your
Extension #6 (pairwise margin ranking loss). Both checkpoints are
needed for a fair four-way comparison later.

**Each cell below checks if its checkpoint already exists and skips
training if so** -- safe to re-run this whole notebook without
wasting compute on checkpoints you already have.

The model architecture (`LTRPredictor`, defined in
`0b_train_predictor.py`) is a custom OPT-125M + linear score head --
NOT a standard HF classification model. The benchmark and evaluation
scripts already account for this; don't swap in
`AutoModelForSequenceClassification` anywhere, that was the bug that
caused the `config.json` errors in earlier sessions.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

import os

ckpt1 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-original')
if os.path.exists(os.path.join(ckpt1, 'pytorch_model.bin')):
    print(f"Checkpoint already exists at {ckpt1}, skipping Run 1.")
else:
    !python scripts/0b_train_predictor.py \
        --train_set ./data/train_predictor.jsonl \
        --output_dir ./checkpoints/opt125m-ltr-original \
        --loss_type listmle \
        --epochs 10

# Verify immediately -- don't wait until later to find out it failed
!ls {ckpt1}

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

import os

ckpt2 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-marginloss')
if os.path.exists(os.path.join(ckpt2, 'pytorch_model.bin')):
    print(f"Checkpoint already exists at {ckpt2}, skipping Run 2.")
else:
    !python scripts/0b_train_predictor.py \
        --train_set ./data/train_predictor.jsonl \
        --output_dir ./checkpoints/opt125m-ltr-marginloss \
        --loss_type margin \
        --margin 1.0 \
        --pair_filter_delta 0.20 \
        --epochs 10

# Verify immediately
!ls {ckpt2}

## 10. FREE TIER ONLY — enable quantization

Skip / leave False if you're on Colab Pro with an A100. Only set True
on a free T4 (16GB) -- Llama-3.1-8B at FP16 alone needs ~16GB, leaving
no headroom for vLLM's KV cache.

In [ ]:
USE_QUANTIZATION = True   # set False on Colab Pro / A100

## 11. Run the benchmark sweep — Dataset 1 (in-distribution)

Compares all four schedulers: FCFS, Classification, Original LTR
(listmle checkpoint), and LTR + Prompt Length (margin-loss checkpoint
+ the prompt-length scoring patch). Expect 20-40 minutes depending on
GPU tier.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

# Pre-flight check -- fail fast with a clear message instead of burning
# GPU time on a run that's doomed to error out partway through.
ckpt1 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-original', 'pytorch_model.bin')
ckpt2 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-marginloss', 'pytorch_model.bin')
dataset1 = os.path.join(repo_path, 'data', 'dataset1_eval.jsonl')
assert os.path.exists(ckpt1), f"Missing checkpoint: {ckpt1} -- re-run Cell 18"
assert os.path.exists(ckpt2), f"Missing checkpoint: {ckpt2} -- re-run Cell 19"
assert os.path.exists(dataset1), f"Missing dataset: {dataset1} -- re-run Cell 16"
print("Pre-flight check passed -- both checkpoints and Dataset 1 found.")

quant_flag = "--quantize" if USE_QUANTIZATION else ""

!python scripts/3_vllm_benchmark.py \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --ltr_predictor_path ./checkpoints/opt125m-ltr-original \
    --ltr_promptlen_predictor_path ./checkpoints/opt125m-ltr-marginloss \
    --dataset ./data/dataset1_eval.jsonl \
    --request_rates 5 10 20 30 40 50 60 \
    --schedulers fcfs classification ltr ltr_promptlen \
    --output result/results_dataset1.json \
    {quant_flag}

## 12. Run the benchmark sweep — Dataset 2 (out-of-distribution)

Reproduces the original paper's Dataset 2 evaluation (their reported
weak spot), now with your extension included as a fourth scheduler.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

ckpt1 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-original', 'pytorch_model.bin')
ckpt2 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-marginloss', 'pytorch_model.bin')
dataset2 = os.path.join(repo_path, 'data', 'dataset2_eval.jsonl')
assert os.path.exists(ckpt1), f"Missing checkpoint: {ckpt1} -- re-run Cell 18"
assert os.path.exists(ckpt2), f"Missing checkpoint: {ckpt2} -- re-run Cell 19"
assert os.path.exists(dataset2), f"Missing dataset: {dataset2} -- re-run Cell 16"
print("Pre-flight check passed -- both checkpoints and Dataset 2 found.")

quant_flag = "--quantize" if USE_QUANTIZATION else ""

!python scripts/3_vllm_benchmark.py \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --ltr_predictor_path ./checkpoints/opt125m-ltr-original \
    --ltr_promptlen_predictor_path ./checkpoints/opt125m-ltr-marginloss \
    --dataset ./data/dataset2_eval.jsonl \
    --request_rates 5 10 20 30 40 50 60 \
    --schedulers fcfs classification ltr ltr_promptlen \
    --output result/results_dataset2.json \
    {quant_flag}

## 13. Confirm results saved (already in Drive — no extra copy step needed)

Since this entire notebook has been running inside
`/content/drive/MyDrive/ltr_extension/prompt-aware-ltr` from Cell 3 onward,
and the previous cell copied them into `result/`, `results_dataset1.json`
and `results_dataset2.json` are ALREADY sitting on your Drive under
`result/`. This cell just confirms that and shows you the Drive path
for downloading them to your Mac.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

import os
for fname in ['results_dataset1.json', 'results_dataset2.json']:
    full_path = os.path.join(repo_path, 'result', fname)
    exists = os.path.exists(full_path)
    print(f"{fname}: {'FOUND' if exists else 'MISSING'} at {full_path}")

## 14. Extended Metrics — Throughput & GPU Utilization

Runs `scripts/5_extended_metrics.py` (added in the PR merge). Computes
throughput (tokens/s, requests/s) from `result/results_dataset1.json` and
`result/results_dataset2.json` for both datasets automatically. Set
`RUN_GPU_PROFILING = True` below to also run a short live GPU memory/
utilization probe per dataset (loads the model again briefly -- requires
GPU + vLLM, adds extra runtime; leave False to skip).

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

RUN_GPU_PROFILING = False  # set True to also probe live GPU memory/utilization
quant_flag = "--quantize" if USE_QUANTIZATION else ""
gpu_flag = "--run_gpu_profiling" if RUN_GPU_PROFILING else ""

!python scripts/5_extended_metrics.py \
    --results_dir ./result \
    --data_dir ./data \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --ltr_predictor_path ./checkpoints/opt125m-ltr-original \
    --ltr_promptlen_predictor_path ./checkpoints/opt125m-ltr-marginloss \
    --output result/extended_metrics.json \
    {gpu_flag} {quant_flag}

## 15. HOL Blocking Analysis

Runs `scripts/6_hol_blocking_eval.py`, estimating Head-of-Line blocking
reduction (p90/mean latency ratio at high load, and latency-vs-request-rate
slope) from the same two result files. CPU only -- no GPU or vLLM needed.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

!python scripts/6_hol_blocking_eval.py \
    --results_dir ./result \
    --data_dir ./data \
    --output result/hol_blocking_metrics.json

## 16. TTFT Benchmark Sweep — Dataset 1

Runs the same four schedulers (FCFS, Classification, Original LTR, LTR +
Prompt Length) across all seven request rates on Dataset 1, using
`scripts/3_vllm_benchmark.py`'s Time-to-First-Token tracking (mean/p90
TTFT, pulled directly from vLLM's own per-request metrics rather than
derived from elapsed/n_tokens). Writes to `ttft_dataset1.json` — a
separate file from `results_dataset1.json`, so the frozen/reported
latency numbers already used in the report are never overwritten.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

# Pre-flight check -- fail fast instead of burning GPU time on a doomed run.
ckpt1 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-original', 'pytorch_model.bin')
ckpt2 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-marginloss', 'pytorch_model.bin')
dataset1 = os.path.join(repo_path, 'data', 'dataset1_eval.jsonl')
assert os.path.exists(ckpt1), f"Missing checkpoint: {ckpt1} -- re-run Cell 18"
assert os.path.exists(ckpt2), f"Missing checkpoint: {ckpt2} -- re-run Cell 19"
assert os.path.exists(dataset1), f"Missing dataset: {dataset1} -- re-run Cell 16"
print("Pre-flight check passed -- both checkpoints and Dataset 1 found.")

quant_flag = "--quantize" if USE_QUANTIZATION else ""

!python scripts/3_vllm_benchmark.py \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --ltr_predictor_path ./checkpoints/opt125m-ltr-original \
    --ltr_promptlen_predictor_path ./checkpoints/opt125m-ltr-marginloss \
    --dataset ./data/dataset1_eval.jsonl \
    --request_rates 5 10 20 30 40 50 60 \
    --schedulers fcfs classification ltr ltr_promptlen \
    --output result/ttft_dataset1.json \
    {quant_flag}

## 17. TTFT Benchmark Sweep — Dataset 2

Same as Section 16, run against Dataset 2 (out-of-distribution). Writes
to `ttft_dataset2.json` — again separate from `results_dataset2.json`.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

ckpt1 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-original', 'pytorch_model.bin')
ckpt2 = os.path.join(repo_path, 'checkpoints', 'opt125m-ltr-marginloss', 'pytorch_model.bin')
dataset2 = os.path.join(repo_path, 'data', 'dataset2_eval.jsonl')
assert os.path.exists(ckpt1), f"Missing checkpoint: {ckpt1} -- re-run Cell 18"
assert os.path.exists(ckpt2), f"Missing checkpoint: {ckpt2} -- re-run Cell 19"
assert os.path.exists(dataset2), f"Missing dataset: {dataset2} -- re-run Cell 16"
print("Pre-flight check passed -- both checkpoints and Dataset 2 found.")

quant_flag = "--quantize" if USE_QUANTIZATION else ""

!python scripts/3_vllm_benchmark.py \
    --model meta-llama/Llama-3.1-8B-Instruct \
    --ltr_predictor_path ./checkpoints/opt125m-ltr-original \
    --ltr_promptlen_predictor_path ./checkpoints/opt125m-ltr-marginloss \
    --dataset ./data/dataset2_eval.jsonl \
    --request_rates 5 10 20 30 40 50 60 \
    --schedulers fcfs classification ltr ltr_promptlen \
    --output result/ttft_dataset2.json \
    {quant_flag}

---
## 18. Kendall's Tau evaluation 


In [ ]:
!python scripts/2_kendalls_tau_eval.py \
    --predictor_path ./checkpoints/opt125m-ltr-original \
    --test_set ./data/dataset2_eval.jsonl \
    --alpha 0.7 --beta 0.3 \
    --output result/tau_results.json

## 19. Evaluation Metrics Figures

Runs every `scripts/plot_*.py` script against the JSON files in `result/`
and displays each resulting PNG inline as it's generated.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
PROJECT_DIR = '/content/drive/MyDrive/ltr_extension'
repo_path = os.path.join(PROJECT_DIR, 'prompt-aware-ltr')
%cd {repo_path}

import base64
from IPython.display import Image, HTML, display

def show_side_by_side(paths, width_pct=48):
    imgs_html = ""
    for p in paths:
        if not os.path.exists(p):
            print(f"[MISSING] {p}")
            continue
        with open(p, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        imgs_html += f'<img src="data:image/png;base64,{b64}" style="width:{width_pct}%; display:inline-block; margin:1%;">'
    if imgs_html:
        display(HTML(f'<div style="display:flex; justify-content:center;">{imgs_html}</div>'))

figure_scripts = [
    ("scripts/plot_latency_vs_request_rate.py", ["result/latency_dataset1.png", "result/latency_dataset2.png"]),
    ("scripts/plot_ttft_vs_request_rate.py", ["result/ttft_dataset1.png", "result/ttft_dataset2.png"]),
    ("scripts/plot_throughput_by_scheduler.py", ["result/throughput_by_scheduler.png"]),
    ("scripts/plot_gpu_utilization.py", ["result/gpu_utilization.png"]),
    ("scripts/plot_hol_blocking.py", ["result/hol_blocking_ratio.png", "result/hol_blocking_slope.png"]),
    ("scripts/plot_kendalls_tau_table.py", ["result/kendalls_tau_table.png"]),
]

for script, outputs in figure_scripts:
    print(f"=== {script} ===")
    !python {script}
    if script == "scripts/plot_latency_vs_request_rate.py":
        # Dataset 1 vs Dataset 2 -- shown side by side for direct comparison.
        show_side_by_side(outputs)
    else:
        for out in outputs:
            if os.path.exists(out):
                display(Image(out))
            else:
                print(f"[MISSING] {out}")